<a href="https://colab.research.google.com/github/FrancescaMicu/3DUPB_Computer_Vision/blob/main/proiect3DUPB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# instalare api-uri
!pip install transformers accelerate bitsandbytes matplotlib

#
GRID_W, GRID_H = 24, 32
# codificarea hartii
# Bird -> zona blocata
# Normal Ladder -> pas inainte
# Start -> baza de inceput
# Exit -> Iesirea
# Beetle -> inamicul care trebuie ocolit cu orice pret (altfel se pierde jocul)
# Portal -> te teleporteaza la cealalta iesire a portalului
# Vines -> pas inainte dublu
# Broken Ladder -> coborare cu un rand mai jos fortata
TILE = {
    0:('.','#1a1a2e','Bird'),
    1:('N','#dfe6e9','Normal Ladder'),
    2:('S','#00b894','Start'),
    3:('E','#d63031','Exit'),
    4:('K','#f39c12','Beetle'),
    5:('P','#8e44ad','Portal'),
    6:('V','#c0392b','Vines'),
    7:('B','#f1c40f','Broken Ladder'),
}

PASSABLE = {1, 2, 3, 4, 5, 6, 7}
COLORS = {k:v[1] for k, v in TILE.items()}
ASCII_MAP = {v[0]:k for k, v in TILE.items()}
VALID_CHARS = set(ASCII_MAP.keys())
print(VALID_CHARS)

print('✅ Config OK — Joc: Climb the Tower — Grid:', GRID_W, 'x', GRID_H)

{'P', 'V', 'N', 'K', 'E', '.', 'B', 'S'}
✅ Config OK — Joc: Climb the Tower — Grid: 24 x 32


In [ ]:
# descarcam LLM-ul

# conectare cu google colab
from google.colab import drive
from huggingface_hub import snapshot_download

drive.mount('/content/drive')

snapshot_download(
    repo_id="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    local_dir="/content/drive/MyDrive/models/smollm2",
    ignore_patterns=["*.msgpack","*.h5","flax_*"],
)

print("✅ Model salvat în Drive!")

Mounted at /content/drive


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

✅ Model salvat în Drive!


In [ ]:
# incarcare in drive
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import drive

drive.mount('/content/drive')
MODEL_ID = "/content/drive/MyDrive/models/smollm2"

print(f'🔄 Încărcare model din Drive...')
print(f'   GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto')
model.eval()
print('✅ Model încărcat din Drive!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Încărcare model din Drive...
   GPU: True
   VRAM: 15.6 GB


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

✅ Model încărcat din Drive!


In [ ]:
# generare nivel folosind smollm2

# pregatirea promptului
SYSTEM = """You are a game level designer. Generate a dungeon level as ASCII art
            Characters:
                        . = Bird
                        N = Normal Ladder
                        S = Start
                        E = Exit
                        K = Beetle
                        P = Portal
                        V = Vines
                        B = Broken Ladder
            Rules:
            - Exactly 32 rows, each row exactly 24 characters
            - Exactly 1 S and 1 E
            - The start S must always be in the last row
            - The bird X has the same purpose as a wall
            - 1-2-3 beetles K
            - 2 portals P
            - 1-2-3 vines V
            - 1-2-3 broken ladders B
            - The majority of the ladders (N and B) must be connected to form a path upwards
            - There can be ladders that are connected horizontaly
            - There has to be less vines (V) than there are normal ladders (N)
            - Output ONLY the 32 lines, no explanation, no numberin
          """
USER = """
        ........................
        .......................E
        .......................N
        .......................N
        ................K......N
        ................N......N
        ................NNNNNNNN
        ................N.......
        ....P...........N.......
        ....N...........B.......
        ....N...........N.......
        ....NNNNNNNNNNNNN.......
        ............N...........
        ............N...........
        ............N....V......
        ............NNNNNN......
        .................N......
        .................N......
        .................N......
        ....P............B......
        ....NNNNNNNNNNNNNN......
        ....N...................
        ....V...................
        ....N...................
        ....NNNNNNK.............
        ..........N.............
        ..........N.............
        ..........NNNNNNN.......
        ................N.......
        ................K.......
        ................N.......
        ................S.......
        Modify this level to make it more interesting. Output ONLY the 32 lines"""

def calling_llm(SYSTEM, USER, temp = 0.4, max_new = 1000):
  msgs = [{'role':'system','content':system}, {'role':'user','content':user}]
  txt  = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
  inp  = tokenizer(txt, return_tensors='pt').to(model.device)
  with torch.no_grad():
      out = model.generate(**inp, max_new_tokens=max_new, temperature=temp,
              do_sample=True, top_p=0.9,
              pad_token_id=tokenizer.eos_token_id)
  return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

def parse_ascii_permisive(resp):
  lines = []
  for raw_line in resp.strip().split('\n'):
    line = raw_line.strip()
    if not line or len(line) < GRID_W / 4:
      continue
    cleaned = ''.join(c if c in VALID_CHARS else '.' for c in line)
    if len(cleaned) < GRID_W / 2:
      continue
    if len(cleaned) > GRID_W:
      cleaned = cleaned[:GRID_W]
    if len(cleaned) < GRID_W:
      pad = '.'
      cleaned = cleaned + pad * (16 - len(cleaned))
      lines.append([ASCII_MAP.get(c, 1) for c in cleaned])
    if len(lines) == GRID_H:
      break
    while len(lines) < GRID_H:
      if len(lines) == 0 or len(lines) == GRID_H - 1:
        lines.append([0] * GRID_W)
      else:
        lines.append([0] + [1]*(GRID_W-2) + [0])

  return lines[:GRID_H]

